### Vechile Damage Detection Hyperparameter Tunning

In [28]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import time
import torchvision.models as models
from matplotlib import pyplot as plt
import optuna

In [29]:
device = torch.device("mps" if torch.mps.is_available else "cpu")
device

device(type='mps')

### Load Data

In [30]:
image_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast= 0.2),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229,0.224,0.225])

]
)

In [31]:
dataset_path = "./dataset"

dataset = datasets.ImageFolder(root=dataset_path, transform=image_transforms)
len(dataset)

2300

In [32]:
classes_names = dataset.classes
classes_names

['F_Breakage', 'F_Crushed', 'F_Normal', 'R_Breakage', 'R_Crushed', 'R_Normal']

In [33]:
num_classes = len(dataset.classes)
num_classes

6

In [34]:
train_size = int(len(dataset)*0.75)
val_size = len(dataset) - train_size

train_size, val_size

(1725, 575)

In [35]:
from torch.utils.data import random_split

In [36]:
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

In [37]:
train_loader = DataLoader(train_dataset, batch_size= 32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True)

Model Training & Hyperparameter Tunning

In [38]:
# Loading the Pretrained ResNet Model
class CarDamageClassifierResNet(nn.Module):
    def __init__(self, num_classes, dropout_rate):
        super().__init__()
        self.model = models.resnet50(weights = "DEFAULT")
        # Freeze all layers except the final fully connected layer

        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze layer 4 and fc layer

        for param in self.model.layer4.parameters():
            param.requires_grad = True

        # Replace the final fully connected layer 
        self.model.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.model.fc.in_features, num_classes)
        )
    def forward(self,x):
        x = self.model(x)
        return x


In [39]:
# define objective function for optuna
def objective(trial):
    # Suggest values for the hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)

    dropout_rate = trial.suggest_float('dropout_rate', 0.2, 0.7)

    # Load model

    model = CarDamageClassifierResNet(num_classes , dropout_rate).to(device)

    # Define the loss function and optmizier

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p : p.requires_grad, model.parameters()), lr= lr)

    # Training Loop (using fewer epochs for faster hyperparameters tuning)
    epochs = 6
    start = time.time()
    
    for epoch in range(epochs):
        model.train()

        running_loss = 0.0

        for batch_num , (images,labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()* images.size(0)

        epoch_loss = running_loss/len(train_loader.dataset)

        # Validation Loop

        model.eval()

        correct, total = 0,0
        with torch.no_grad():
            for images , labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)

                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        accuracy = 100 * correct/total

        # Report intermediate result to Optuna
        trial.report(accuracy, epoch)

        # Handle pruning (if applicable)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        
    end = time.time()

    print(f"Execution time: {end - start} seconds")

    return accuracy
            



In [40]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

[I 2026-04-02 08:22:18,697] A new study created in memory with name: no-name-e27aba48-c6f2-4eb5-aa15-bae75fc80242
[I 2026-04-02 09:44:12,235] Trial 0 finished with value: 73.73913043478261 and parameters: {'lr': 2.4991177491303147e-05, 'dropout_rate': 0.5356030788965387}. Best is trial 0 with value: 73.73913043478261.


Execution time: 4913.128660917282 seconds


[I 2026-04-02 09:49:54,958] Trial 1 finished with value: 76.69565217391305 and parameters: {'lr': 2.532028014263873e-05, 'dropout_rate': 0.44479351772253384}. Best is trial 1 with value: 76.69565217391305.


Execution time: 342.33589482307434 seconds


[I 2026-04-02 09:55:24,824] Trial 2 finished with value: 67.82608695652173 and parameters: {'lr': 1.4346979746366698e-05, 'dropout_rate': 0.49599624730033504}. Best is trial 1 with value: 76.69565217391305.


...

[I 2026-04-02 10:59:17,809] Trial 16 finished with value: 79.1304347826087 and parameters: {'lr': 0.0010537358994495564, 'dropout_rate': 0.5742451214298138}. Best is trial 14 with value: 81.04347826086956.


Execution time: 344.58340096473694 seconds


[I 2026-04-02 11:05:06,446] Trial 17 finished with value: 81.21739130434783 and parameters: {'lr': 0.00029998576473226775, 'dropout_rate': 0.6593149255258369}. Best is trial 17 with value: 81.21739130434783.


Execution time: 348.27954292297363 seconds


[I 2026-04-02 11:06:02,516] Trial 18 pruned. 
[I 2026-04-02 11:17:30,483] Trial 19 pruned. 


In [41]:
study.best_params

{'lr': 0.00029998576473226775, 'dropout_rate': 0.6593149255258369}

After training the model for 6 epochs the best parameters we found for learning_rate:0.00029998576473226775 and dropout_rate :  ç593149255258369 